# 04 — Gene Lookup

HGNC and Ensembl BioMart reference files and columns before joining.

In [ ]:
import sys, os

_notebook_dir = os.path.dirname(os.path.abspath('__file__'))
sys.path.insert(0, os.path.join(_notebook_dir, '..', 'scripts'))

import pandas as pd
from data_utils import REF

print("REF path:", REF)

In [ ]:
# Load HGNC
hgnc = pd.read_csv(os.path.join(REF, 'hgnc_complete.txt'), sep='\t', low_memory=False)
print('HGNC shape:', hgnc.shape)
print('HGNC columns:', hgnc.columns.tolist())
print()
print(hgnc.head(3))

In [ ]:
# Load Ensembl BioMart
ensembl = pd.read_csv(os.path.join(REF, 'ensembl_biomart.txt'), sep='\t', low_memory=False)
print('Ensembl shape:', ensembl.shape)
print('Ensembl columns:', ensembl.columns.tolist())
print()
print(ensembl.head(3))

In [ ]:
# Step 1: Inner join HGNC to Ensembl on ENSG ID
gene_lookup = hgnc.merge(
    ensembl,
    left_on="Ensembl gene ID",
    right_on="Gene stable ID",
    how="inner"
)

# Step 2: Filter to protein_coding only
gene_lookup = gene_lookup[gene_lookup["Gene type"] == "protein_coding"].copy()

# Step 3: Rename columns to contract spec
gene_lookup = gene_lookup.rename(columns={
    "Ensembl gene ID":  "ensg_id",
    "Approved symbol":  "hgnc_symbol",
    "Previous symbols": "prev_symbols",
    "Alias symbols":    "alias_symbols",
    "HGNC ID":          "hgnc_id",
    "Gene type":        "biotype",
})

# Step 4: Drop columns no longer needed
gene_lookup = gene_lookup.drop(columns=["Gene stable ID", "Locus group"])

# Step 5: Convert comma-delimited lists → pipe-delimited (contract rule)
for col in ["prev_symbols", "alias_symbols"]:
    gene_lookup[col] = (
        gene_lookup[col]
        .fillna("")
        .str.strip()
        .str.replace(r",\s*", "|", regex=True)
        .replace("", float("nan"))
    )

# Step 6: Strip any ENSG version suffixes (precaution)
gene_lookup["ensg_id"] = gene_lookup["ensg_id"].str.replace(
    r"\.\d+$", "", regex=True
)

# Step 7: Final column order per contract
gene_lookup = gene_lookup[[
    "ensg_id", "hgnc_symbol", "prev_symbols",
    "alias_symbols", "hgnc_id", "biotype"
]]

# Force plain object dtype — prevents Arrow-backed StringDtype from showing as 'str'
gene_lookup = gene_lookup.astype(object)

print(gene_lookup.shape)
print(gene_lookup.dtypes)
print(gene_lookup.head(5))

In [ ]:
print(gene_lookup.dtypes)

In [ ]:
print("=== VALIDATION ===")
print(f"Total rows:           {len(gene_lookup):,}")
print(f"Null ensg_id:         {gene_lookup['ensg_id'].isna().sum()}")
print(f"Duplicate ensg_id:    {gene_lookup['ensg_id'].duplicated().sum()}")
print(f"Null hgnc_symbol:     {gene_lookup['hgnc_symbol'].isna().sum()}")
print(f"Unique biotypes:      {gene_lookup['biotype'].unique()}")
print(f"Has prev_symbols:     {gene_lookup['prev_symbols'].notna().sum():,}")
print(f"Has alias_symbols:    {gene_lookup['alias_symbols'].notna().sum():,}")

In [ ]:
OUT = os.path.join(REF, "gene_lookup.parquet")

gene_lookup.to_parquet(OUT, index=False, engine="fastparquet")

# Confirm it saved correctly by reading it back
confirm = pd.read_parquet(OUT, engine="fastparquet")
print(f"Saved and verified: {confirm.shape}")
print(f"File size: {os.path.getsize(OUT):,} bytes")

In [ ]:
# Check for duplicates and non-string column names
print("Duplicate columns:", gene_lookup.columns[gene_lookup.columns.duplicated()].tolist())
print("Column dtypes (name types):", set(type(c) for c in gene_lookup.columns))
print("Any NaN column names:", gene_lookup.columns.isna().any())
